# 03 - ECG Feature Engineering and Experimental Input Construction

### 3.1. Objective

Extend the canonical heartbeat dataset created in Notebook 02 by
calculating eight explicit morphology features from each standardized
255-sample ECG heartbeat.

The resulting feature components will support the same four progressive
input stages for all model families:

1. ECG
2. ECG + 6RR
3. ECG + 13Context
4. ECG + 21 engineered features

The four stages will later be evaluated using:

- Logistic Regression
- Random Forest
- XGBoost
- Convolutional Neural Networks

### Feature Composition

The 13 context features were already created in Notebook 02:

- 6 immediate RR features
- 3 rhythm-history features
- 4 morphology-context features

This notebook adds 8 explicit morphology statistics calculated directly
from the standardized heartbeat waveform.

Therefore:

13 context features + 8 morphology features = 21 engineered features

The heartbeat rows, labels, record assignments, and fixed cross-validation
folds are not modified in this notebook.

### 3.2. Load the Canonical Dataset from Notebook 02

Load the standardized ECG heartbeat waveforms, six-RR feature set,
13-context feature set, labels, heartbeat metadata, and fixed DS1
cross-validation folds generated in Notebook 02.

All subsequent feature matrices must preserve the exact same heartbeat
rows and row ordering.

### Load processed dataset

In [1]:
# ============================================================
# SECTION 1: IMPORT LIBRARIES
# ============================================================

from pathlib import Path

import numpy as np
import pandas as pd

from scipy.stats import skew, kurtosis


# ============================================================
# SECTION 2: FIND PROJECT ROOT
# ============================================================

CURRENT_DIR = Path.cwd().resolve()

PROJECT_ROOT = CURRENT_DIR


while not (
    PROJECT_ROOT
    / "requirements.txt"
).exists():

    if (
        PROJECT_ROOT.parent
        == PROJECT_ROOT
    ):

        raise FileNotFoundError(
            "Could not locate the ECG-ML project root."
        )

    PROJECT_ROOT = (
        PROJECT_ROOT.parent
    )


# ============================================================
# SECTION 3: DEFINE PROCESSED DATA DIRECTORY
# ============================================================

PROCESSED_DATA_DIR = (
    PROJECT_ROOT
    / "data"
    / "processed"
)


# ============================================================
# SECTION 4: LOAD CANONICAL ECG WAVEFORMS
# ============================================================

X_ds1_ecg = np.load(
    PROCESSED_DATA_DIR
    / "X_ds1_ecg.npy"
)


X_ds2_ecg = np.load(
    PROCESSED_DATA_DIR
    / "X_ds2_ecg.npy"
)


# ============================================================
# SECTION 5: LOAD SIX-RR FEATURE MATRICES
# ============================================================

X_ds1_rr6 = np.load(
    PROCESSED_DATA_DIR
    / "X_ds1_rr6.npy"
)


X_ds2_rr6 = np.load(
    PROCESSED_DATA_DIR
    / "X_ds2_rr6.npy"
)


# ============================================================
# SECTION 6: LOAD THIRTEEN-CONTEXT FEATURE MATRICES
# ============================================================

X_ds1_context13 = np.load(
    PROCESSED_DATA_DIR
    / "X_ds1_context13.npy"
)


X_ds2_context13 = np.load(
    PROCESSED_DATA_DIR
    / "X_ds2_context13.npy"
)


# ============================================================
# SECTION 7: LOAD FEATURE NAMES
# ============================================================

rr6_feature_names = (
    np.load(
        PROCESSED_DATA_DIR
        / "rr6_feature_names.npy"
    )
    .astype(str)
    .tolist()
)


context13_feature_names = (
    np.load(
        PROCESSED_DATA_DIR
        / "context13_feature_names.npy"
    )
    .astype(str)
    .tolist()
)


# ============================================================
# SECTION 8: LOAD LABELS
# ============================================================

y_ds1 = np.load(
    PROCESSED_DATA_DIR
    / "y_ds1.npy"
)


y_ds2 = np.load(
    PROCESSED_DATA_DIR
    / "y_ds2.npy"
)


# ============================================================
# SECTION 9: LOAD CANONICAL METADATA
# ============================================================

ds1_metadata = pd.read_csv(
    PROCESSED_DATA_DIR
    / "ds1_metadata.csv",
    dtype={
        "beat_uid": str,
        "record_id": str,
    }
)


ds2_metadata = pd.read_csv(
    PROCESSED_DATA_DIR
    / "ds2_metadata.csv",
    dtype={
        "beat_uid": str,
        "record_id": str,
    }
)


# ============================================================
# SECTION 10: LOAD FIXED DS1 RECORD FOLDS
# ============================================================

FOLD_PATH = (
    PROCESSED_DATA_DIR
    / "ds1_cv_record_folds.csv"
)


if not FOLD_PATH.exists():

    raise FileNotFoundError(
        "Fixed DS1 fold file was not found. "
        "Run the corrected Notebook 02 first."
    )


ds1_record_folds = pd.read_csv(
    FOLD_PATH,
    dtype={
        "record_id": str,
        "cv_fold": int,
    }
)


# ============================================================
# SECTION 11: EXPECTED SIX-RR FEATURE ORDER
# ============================================================

EXPECTED_RR6_FEATURE_NAMES = [

    "rr_pre_sec",

    "rr_post_sec",

    "rr_local_median_sec",

    "rr_pre_norm",

    "rr_post_norm",

    "rr_pre_post_ratio",
]


# ============================================================
# SECTION 12: EXPECTED THIRTEEN-CONTEXT FEATURE ORDER
# ============================================================

EXPECTED_CONTEXT13_FEATURE_NAMES = [

    # --------------------------------------------------------
    # Six immediate RR features
    # --------------------------------------------------------

    "rr_pre_sec",

    "rr_post_sec",

    "rr_local_median_sec",

    "rr_pre_norm",

    "rr_post_norm",

    "rr_pre_post_ratio",


    # --------------------------------------------------------
    # Three rhythm-history features
    # --------------------------------------------------------

    "rr_history_mean_sec",

    "rr_history_std_sec",

    "rr_history_cv",


    # --------------------------------------------------------
    # Four morphology-context features
    # --------------------------------------------------------

    "morph_corr_prev",

    "morph_dist_prev",

    "morph_corr_template",

    "morph_dist_template",
]


# ============================================================
# SECTION 13: VERIFY BASIC DIMENSIONS
# ============================================================

assert (
    X_ds1_ecg.ndim
    == 2
)

assert (
    X_ds2_ecg.ndim
    == 2
)


assert (
    X_ds1_ecg.shape[1]
    == 255
)

assert (
    X_ds2_ecg.shape[1]
    == 255
)


assert (
    X_ds1_rr6.shape[1]
    == 6
)

assert (
    X_ds2_rr6.shape[1]
    == 6
)


assert (
    X_ds1_context13.shape[1]
    == 13
)

assert (
    X_ds2_context13.shape[1]
    == 13
)


# ============================================================
# SECTION 14: VERIFY DS1 ROW ALIGNMENT
# ============================================================

assert (
    len(X_ds1_ecg)
    == len(X_ds1_rr6)
    == len(X_ds1_context13)
    == len(y_ds1)
    == len(ds1_metadata)
)


# ============================================================
# SECTION 15: VERIFY DS2 ROW ALIGNMENT
# ============================================================

assert (
    len(X_ds2_ecg)
    == len(X_ds2_rr6)
    == len(X_ds2_context13)
    == len(y_ds2)
    == len(ds2_metadata)
)


# ============================================================
# SECTION 16: VERIFY HEARTBEAT IDENTIFIERS
# ============================================================

assert (
    ds1_metadata[
        "beat_uid"
    ]
    .is_unique
)


assert (
    ds2_metadata[
        "beat_uid"
    ]
    .is_unique
)


beat_uid_overlap = (
    set(
        ds1_metadata[
            "beat_uid"
        ]
    )
    .intersection(
        set(
            ds2_metadata[
                "beat_uid"
            ]
        )
    )
)


assert (
    len(
        beat_uid_overlap
    )
    == 0
)


# ============================================================
# SECTION 17: VERIFY LABEL ALIGNMENT
# ============================================================

assert np.array_equal(
    y_ds1,
    ds1_metadata[
        "class"
    ]
    .to_numpy()
)


assert np.array_equal(
    y_ds2,
    ds2_metadata[
        "class"
    ]
    .to_numpy()
)


# ============================================================
# SECTION 18: VERIFY FEATURE NAME ORDER
# ============================================================

assert (
    rr6_feature_names
    == EXPECTED_RR6_FEATURE_NAMES
)


assert (
    context13_feature_names
    == EXPECTED_CONTEXT13_FEATURE_NAMES
)


# ============================================================
# SECTION 19: VERIFY RR6 IS EXACTLY THE FIRST SIX CONTEXT13 COLUMNS
# ============================================================

assert np.allclose(
    X_ds1_rr6,
    X_ds1_context13[
        :,
        :6
    ],
    equal_nan=True
)


assert np.allclose(
    X_ds2_rr6,
    X_ds2_context13[
        :,
        :6
    ],
    equal_nan=True
)


# ============================================================
# SECTION 20: VERIFY FIXED FOLDS
# ============================================================

required_ds1_columns = {
    "beat_uid",
    "record_id",
    "class",
    "cv_fold",
}


missing_ds1_columns = (
    required_ds1_columns
    - set(
        ds1_metadata.columns
    )
)


assert not missing_ds1_columns, (
    "Missing required DS1 metadata columns: "
    f"{missing_ds1_columns}"
)


fold_map = dict(
    zip(
        ds1_record_folds[
            "record_id"
        ],
        ds1_record_folds[
            "cv_fold"
        ]
    )
)


expected_row_folds = (
    ds1_metadata[
        "record_id"
    ]
    .map(
        fold_map
    )
)


assert (
    expected_row_folds
    .notna()
    .all()
)


assert np.array_equal(
    ds1_metadata[
        "cv_fold"
    ]
    .to_numpy(
        dtype=int
    ),

    expected_row_folds
    .to_numpy(
        dtype=int
    )
)


# ============================================================
# SECTION 21: DISPLAY LOADED DATA
# ============================================================

print(
    "DS1 ECG:",
    X_ds1_ecg.shape
)

print(
    "DS1 RR6:",
    X_ds1_rr6.shape
)

print(
    "DS1 Context13:",
    X_ds1_context13.shape
)


print(
    "\nDS2 ECG:",
    X_ds2_ecg.shape
)

print(
    "DS2 RR6:",
    X_ds2_rr6.shape
)

print(
    "DS2 Context13:",
    X_ds2_context13.shape
)


print(
    "\nFixed DS1 folds:"
)

display(
    ds1_record_folds
)


print(
    "\nCanonical dataset and fixed folds verified."
)

DS1 ECG: (50992, 255)
DS1 RR6: (50992, 6)
DS1 Context13: (50992, 13)

DS2 ECG: (49686, 255)
DS2 RR6: (49686, 6)
DS2 Context13: (49686, 13)

Fixed DS1 folds:


,record_id,cv_fold
0,114,1
1,115,1
2,119,1
3,124,1
4,203,1
5,101,2
6,109,2
7,118,2
8,201,2
9,207,2



Canonical dataset and fixed folds verified.


### 3.3. Extract Eight Explicit Morphology Features

Eight scalar morphology descriptors are calculated directly from each
standardized 255-sample heartbeat waveform.

These descriptors characterize the current heartbeat itself and are
different from the morphology-context features created in Notebook 02,
which compare the current heartbeat with previous beats or a local
template.

The eight explicit morphology features are:

1. Minimum amplitude
2. Maximum amplitude
3. Peak-to-peak amplitude
4. Mean absolute amplitude
5. Skewness
6. Kurtosis
7. Mean absolute first difference
8. Maximum absolute first difference

Because each heartbeat was already z-score normalized in Notebook 02,
mean, standard deviation, RMS amplitude, and total energy are not retained
because they are constant or nearly constant by construction.

In [2]:
# ============================================================
# EIGHT EXPLICIT MORPHOLOGY FEATURE NAMES
# ============================================================

# The ECG heartbeat waveforms were already independently
# z-score normalized in Notebook 02.
#
# Therefore, waveform mean, standard deviation, RMS, and
# total squared amplitude are not retained because they
# become constant or nearly constant by construction.

MORPHOLOGY8_FEATURE_NAMES = [

    "minimum_amplitude",

    "maximum_amplitude",

    "peak_to_peak",

    "mean_absolute_amplitude",

    "skewness",

    "kurtosis",

    "mean_absolute_derivative",

    "maximum_absolute_derivative",
]


# ============================================================
# EXTRACT EIGHT MORPHOLOGY FEATURES
# ============================================================

def extract_morphology8_features(
    beats
):
    """
    Extract eight scalar morphology descriptors from each
    standardized 255-sample ECG heartbeat.

    Parameters
    ----------
    beats : np.ndarray
        Shape (n_beats, 255).

    Returns
    -------
    feature_matrix : np.ndarray
        Shape (n_beats, 8).
    """

    feature_rows = []


    # ========================================================
    # PROCESS EACH HEARTBEAT
    # ========================================================

    for beat in beats:

        # ----------------------------------------------------
        # Verify heartbeat dimensionality
        # ----------------------------------------------------

        if beat.ndim != 1:

            raise ValueError(
                "Each heartbeat must be one-dimensional."
            )


        if len(
            beat
        ) != 255:

            raise ValueError(
                f"Expected 255 ECG samples, "
                f"got {len(beat)}."
            )


        # ====================================================
        # AMPLITUDE / MORPHOLOGY FEATURES
        # ====================================================

        minimum_amplitude = float(
            np.min(
                beat
            )
        )


        maximum_amplitude = float(
            np.max(
                beat
            )
        )


        peak_to_peak = float(
            np.ptp(
                beat
            )
        )


        mean_absolute_amplitude = float(
            np.mean(
                np.abs(
                    beat
                )
            )
        )


        # ====================================================
        # DISTRIBUTION-SHAPE FEATURES
        # ====================================================

        skewness = float(
            skew(
                beat,
                bias=False
            )
        )


        kurt = float(
            kurtosis(
                beat,
                fisher=True,
                bias=False
            )
        )


        # ====================================================
        # FIRST-DIFFERENCE FEATURES
        # ====================================================

        first_difference = np.diff(
            beat
        )


        mean_absolute_derivative = float(
            np.mean(
                np.abs(
                    first_difference
                )
            )
        )


        maximum_absolute_derivative = float(
            np.max(
                np.abs(
                    first_difference
                )
            )
        )


        # ====================================================
        # STORE FEATURE ROW
        # ====================================================

        feature_rows.append(
            [
                minimum_amplitude,
                maximum_amplitude,
                peak_to_peak,
                mean_absolute_amplitude,
                skewness,
                kurt,
                mean_absolute_derivative,
                maximum_absolute_derivative,
            ]
        )


    # ========================================================
    # RETURN FEATURE MATRIX
    # ========================================================

    feature_matrix = np.asarray(
        feature_rows,
        dtype=np.float32
    )


    return feature_matrix


# ============================================================
# VERIFY FEATURE COUNT
# ============================================================

assert (
    len(
        MORPHOLOGY8_FEATURE_NAMES
    )
    == 8
)

### 3.4. Generate the Eight-Feature Morphology Matrices

Calculate the same eight explicit morphology descriptors for every
canonical DS1 and DS2 heartbeat.

No heartbeat rows are added, removed, shuffled, or reordered.

In [3]:
# ============================================================
# EXTRACT DS1 MORPHOLOGY8 FEATURES
# ============================================================

X_ds1_morphology8 = (
    extract_morphology8_features(
        X_ds1_ecg
    )
)


# ============================================================
# EXTRACT DS2 MORPHOLOGY8 FEATURES
# ============================================================

X_ds2_morphology8 = (
    extract_morphology8_features(
        X_ds2_ecg
    )
)


# ============================================================
# VERIFY ROW ALIGNMENT
# ============================================================

assert (
    X_ds1_morphology8.shape[0]
    == X_ds1_ecg.shape[0]
    == len(y_ds1)
    == len(ds1_metadata)
)


assert (
    X_ds2_morphology8.shape[0]
    == X_ds2_ecg.shape[0]
    == len(y_ds2)
    == len(ds2_metadata)
)


# ============================================================
# VERIFY FEATURE DIMENSIONS
# ============================================================

assert (
    X_ds1_morphology8.shape[1]
    == 8
)


assert (
    X_ds2_morphology8.shape[1]
    == 8
)


assert (
    X_ds1_morphology8.shape[1]
    == len(
        MORPHOLOGY8_FEATURE_NAMES
    )
)


assert (
    X_ds2_morphology8.shape[1]
    == len(
        MORPHOLOGY8_FEATURE_NAMES
    )
)


# ============================================================
# VERIFY THERE ARE NO INFINITE VALUES
# ============================================================

assert not np.isinf(
    X_ds1_morphology8
).any()


assert not np.isinf(
    X_ds2_morphology8
).any()


# ============================================================
# REPORT MISSING MORPHOLOGY VALUES, IF ANY
# ============================================================

ds1_morphology_nan_count = int(
    np.isnan(
        X_ds1_morphology8
    )
    .sum()
)


ds2_morphology_nan_count = int(
    np.isnan(
        X_ds2_morphology8
    )
    .sum()
)


print(
    "DS1 Morphology8:",
    X_ds1_morphology8.shape
)

print(
    "DS2 Morphology8:",
    X_ds2_morphology8.shape
)


print(
    "\nDS1 morphology NaN values:",
    ds1_morphology_nan_count
)

print(
    "DS2 morphology NaN values:",
    ds2_morphology_nan_count
)


print(
    "\nMorphology8 features:"
)


for (
    index,
    feature_name

) in enumerate(
    MORPHOLOGY8_FEATURE_NAMES,
    start=1
):

    print(
        f"{index:02d}. "
        f"{feature_name}"
    )

DS1 Morphology8: (50992, 8)
DS2 Morphology8: (49686, 8)

DS1 morphology NaN values: 0
DS2 morphology NaN values: 0

Morphology8 features:
01. minimum_amplitude
02. maximum_amplitude
03. peak_to_peak
04. mean_absolute_amplitude
05. skewness
06. kurtosis
07. mean_absolute_derivative
08. maximum_absolute_derivative


### 3.5. Construct the Complete 21-Feature Representation

The complete engineered feature vector consists of:

- 6 immediate RR features
- 3 rhythm-history features
- 4 morphology-context features
- 8 explicit morphology features

Therefore:

6 + 3 + 4 + 8 = 21 engineered features

These engineered features are combined with the 255-sample ECG waveform
only when a model experiment is constructed.

The four experimental stages are:

### Stage 1
ECG only

### Stage 2
ECG + 6RR

### Stage 3
ECG + 13Context

### Stage 4
ECG + 21 engineered features

No new heartbeat rows or cross-validation folds are created here.

In [4]:
# ============================================================
# SECTION 1: DEFINE COMPLETE 21-FEATURE ORDER
# ============================================================

# The first 13 columns preserve the exact Context13 order
# generated in Notebook 02.
#
# The final 8 columns contain the explicit morphology
# statistics calculated in this notebook.
#
# This ordering makes the progression explicit:
#
# Context13 + Morphology8 = Engineered21

ENGINEERED21_FEATURE_NAMES = (

    context13_feature_names

    + MORPHOLOGY8_FEATURE_NAMES
)


assert (
    len(
        ENGINEERED21_FEATURE_NAMES
    )
    == 21
)


# ============================================================
# SECTION 2: BUILD COMPLETE DS1 ENGINEERED21 MATRIX
# ============================================================

X_ds1_engineered21 = np.column_stack(
    [
        X_ds1_context13,
        X_ds1_morphology8,
    ]
).astype(
    np.float32
)


# ============================================================
# SECTION 3: BUILD COMPLETE DS2 ENGINEERED21 MATRIX
# ============================================================

X_ds2_engineered21 = np.column_stack(
    [
        X_ds2_context13,
        X_ds2_morphology8,
    ]
).astype(
    np.float32
)


# ============================================================
# SECTION 4: VERIFY COMPLETE FEATURE DIMENSIONS
# ============================================================

assert (
    X_ds1_engineered21.shape
    == (
        len(
            X_ds1_ecg
        ),
        21,
    )
)


assert (
    X_ds2_engineered21.shape
    == (
        len(
            X_ds2_ecg
        ),
        21,
    )
)


# ============================================================
# SECTION 5: VERIFY CONTEXT13 WAS PRESERVED EXACTLY
# ============================================================

assert np.allclose(
    X_ds1_engineered21[
        :,
        :13
    ],
    X_ds1_context13,
    equal_nan=True
)


assert np.allclose(
    X_ds2_engineered21[
        :,
        :13
    ],
    X_ds2_context13,
    equal_nan=True
)


# ============================================================
# SECTION 6: VERIFY MORPHOLOGY8 WAS PRESERVED EXACTLY
# ============================================================

assert np.allclose(
    X_ds1_engineered21[
        :,
        13:
    ],
    X_ds1_morphology8,
    equal_nan=True
)


assert np.allclose(
    X_ds2_engineered21[
        :,
        13:
    ],
    X_ds2_morphology8,
    equal_nan=True
)


# ============================================================
# SECTION 7: DEFINE THE FOUR FINAL EXPERIMENTAL STAGES
# ============================================================

EXPERIMENT_STAGE_NAMES = [

    "ECG",

    "ECG+6RR",

    "ECG+13Context",

    "ECG+21",
]


# ============================================================
# SECTION 8: DEFINE STAGE DIMENSIONS
# ============================================================

experiment_stage_manifest = pd.DataFrame(
    [
        {
            "stage":
                1,

            "stage_name":
                "ECG",

            "ecg_dimensions":
                255,

            "auxiliary_dimensions":
                0,

            "tabular_total_dimensions":
                255,

            "auxiliary_features":
                "None",
        },

        {
            "stage":
                2,

            "stage_name":
                "ECG+6RR",

            "ecg_dimensions":
                255,

            "auxiliary_dimensions":
                6,

            "tabular_total_dimensions":
                261,

            "auxiliary_features":
                "6 immediate RR features",
        },

        {
            "stage":
                3,

            "stage_name":
                "ECG+13Context",

            "ecg_dimensions":
                255,

            "auxiliary_dimensions":
                13,

            "tabular_total_dimensions":
                268,

            "auxiliary_features":
                (
                    "6 RR + 3 rhythm-history "
                    "+ 4 morphology-context"
                ),
        },

        {
            "stage":
                4,

            "stage_name":
                "ECG+21",

            "ecg_dimensions":
                255,

            "auxiliary_dimensions":
                21,

            "tabular_total_dimensions":
                276,

            "auxiliary_features":
                (
                    "13 context + "
                    "8 explicit morphology"
                ),
        },
    ]
)


# ============================================================
# SECTION 9: VERIFY TABULAR STAGE DIMENSIONS
# ============================================================

X_ds1_stage1_check = (
    X_ds1_ecg
)


X_ds1_stage2_check = np.column_stack(
    [
        X_ds1_ecg,
        X_ds1_rr6,
    ]
)


X_ds1_stage3_check = np.column_stack(
    [
        X_ds1_ecg,
        X_ds1_context13,
    ]
)


X_ds1_stage4_check = np.column_stack(
    [
        X_ds1_ecg,
        X_ds1_engineered21,
    ]
)


assert (
    X_ds1_stage1_check.shape[1]
    == 255
)


assert (
    X_ds1_stage2_check.shape[1]
    == 261
)


assert (
    X_ds1_stage3_check.shape[1]
    == 268
)


assert (
    X_ds1_stage4_check.shape[1]
    == 276
)


# ============================================================
# SECTION 10: DISPLAY EXPERIMENTAL STAGES
# ============================================================

display(
    experiment_stage_manifest
)


print(
    "\nDS1 stage dimensions:"
)

print(
    "Stage 1 - ECG:",
    X_ds1_stage1_check.shape
)

print(
    "Stage 2 - ECG + 6RR:",
    X_ds1_stage2_check.shape
)

print(
    "Stage 3 - ECG + 13Context:",
    X_ds1_stage3_check.shape
)

print(
    "Stage 4 - ECG + 21:",
    X_ds1_stage4_check.shape
)


# ============================================================
# SECTION 11: RELEASE TEMPORARY CONCATENATED CHECK MATRICES
# ============================================================

# These matrices are deliberately not saved here because
# they duplicate the large ECG waveform arrays.
#
# Each model notebook will construct the required stage
# from the canonical components.

del X_ds1_stage1_check

del X_ds1_stage2_check

del X_ds1_stage3_check

del X_ds1_stage4_check


# ============================================================
# IMPORTANT
# ============================================================

print(
    "\nNaN context values are intentionally retained."
)

print(
    "Imputation must be fitted only on each training fold "
    "during model development."
)

,stage,stage_name,ecg_dimensions,auxiliary_dimensions,tabular_total_dimensions,auxiliary_features
0,1,ECG,255,0,255,None
1,2,ECG+6RR,255,6,261,6 immediate RR features
2,3,ECG+13Context,255,13,268,6 RR + 3 rhythm-history + 4 morphology-context
3,4,ECG+21,255,21,276,13 context + 8 explicit morphology



DS1 stage dimensions:
Stage 1 - ECG: (50992, 255)
Stage 2 - ECG + 6RR: (50992, 261)
Stage 3 - ECG + 13Context: (50992, 268)
Stage 4 - ECG + 21: (50992, 276)

NaN context values are intentionally retained.
Imputation must be fitted only on each training fold during model development.


### 3.6. Save Morphology8 and Engineered21 Components

Save the eight morphology features and the complete 21-feature engineered
representation.

The canonical ECG, RR6, Context13, labels, metadata, and fixed folds
remain those generated in Notebook 02.

The four full experimental input stages will be assembled in the model
notebooks without duplicating the ECG waveform files.

In [5]:
# ============================================================
# SECTION 1: SAVE MORPHOLOGY8 MATRICES
# ============================================================

np.save(
    PROCESSED_DATA_DIR
    / "X_ds1_morphology8.npy",

    X_ds1_morphology8
)


np.save(
    PROCESSED_DATA_DIR
    / "X_ds2_morphology8.npy",

    X_ds2_morphology8
)


# ============================================================
# SECTION 2: SAVE COMPLETE ENGINEERED21 MATRICES
# ============================================================

np.save(
    PROCESSED_DATA_DIR
    / "X_ds1_engineered21.npy",

    X_ds1_engineered21
)


np.save(
    PROCESSED_DATA_DIR
    / "X_ds2_engineered21.npy",

    X_ds2_engineered21
)


# ============================================================
# SECTION 3: SAVE FEATURE NAMES
# ============================================================

np.save(
    PROCESSED_DATA_DIR
    / "morphology8_feature_names.npy",

    np.asarray(
        MORPHOLOGY8_FEATURE_NAMES,
        dtype=str
    )
)


np.save(
    PROCESSED_DATA_DIR
    / "engineered21_feature_names.npy",

    np.asarray(
        ENGINEERED21_FEATURE_NAMES,
        dtype=str
    )
)


# ============================================================
# SECTION 4: DEFINE FEATURE GROUPS
# ============================================================

RHYTHM_HISTORY_FEATURE_NAMES = [

    "rr_history_mean_sec",

    "rr_history_std_sec",

    "rr_history_cv",
]


MORPHOLOGY_CONTEXT_FEATURE_NAMES = [

    "morph_corr_prev",

    "morph_dist_prev",

    "morph_corr_template",

    "morph_dist_template",
]


# ============================================================
# SECTION 5: BUILD FEATURE GROUP MANIFEST
# ============================================================

feature_group_rows = []


# ------------------------------------------------------------
# Six RR features
# ------------------------------------------------------------

for feature_name in rr6_feature_names:

    feature_group_rows.append(
        {
            "feature_name":
                feature_name,

            "feature_group":
                "rr_timing",

            "source_notebook":
                "02_preprocessing",
        }
    )


# ------------------------------------------------------------
# Three rhythm-history features
# ------------------------------------------------------------

for feature_name in RHYTHM_HISTORY_FEATURE_NAMES:

    feature_group_rows.append(
        {
            "feature_name":
                feature_name,

            "feature_group":
                "rhythm_history",

            "source_notebook":
                "02_preprocessing",
        }
    )


# ------------------------------------------------------------
# Four morphology-context features
# ------------------------------------------------------------

for feature_name in MORPHOLOGY_CONTEXT_FEATURE_NAMES:

    feature_group_rows.append(
        {
            "feature_name":
                feature_name,

            "feature_group":
                "morphology_context",

            "source_notebook":
                "02_preprocessing",
        }
    )


# ------------------------------------------------------------
# Eight explicit morphology statistics
# ------------------------------------------------------------

for feature_name in MORPHOLOGY8_FEATURE_NAMES:

    feature_group_rows.append(
        {
            "feature_name":
                feature_name,

            "feature_group":
                "explicit_morphology",

            "source_notebook":
                "03_feature_engineering",
        }
    )


feature_group_manifest = pd.DataFrame(
    feature_group_rows
)


# ============================================================
# SECTION 6: SAVE FEATURE GROUP MANIFEST
# ============================================================

feature_group_manifest.to_csv(
    PROCESSED_DATA_DIR
    / "engineered21_feature_manifest.csv",
    index=False
)


# ============================================================
# SECTION 7: SAVE EXPERIMENT STAGE MANIFEST
# ============================================================

experiment_stage_manifest.to_csv(
    PROCESSED_DATA_DIR
    / "experiment_stage_manifest.csv",
    index=False
)


# ============================================================
# SECTION 8: DISPLAY SAVE SUMMARY
# ============================================================

print(
    "Feature-engineering outputs saved successfully."
)


print(
    "\nMorphology8 feature count:",
    len(
        MORPHOLOGY8_FEATURE_NAMES
    )
)


print(
    "Engineered21 feature count:",
    len(
        ENGINEERED21_FEATURE_NAMES
    )
)


print(
    "\nSaved experimental stages:"
)


display(
    experiment_stage_manifest
)

Feature-engineering outputs saved successfully.

Morphology8 feature count: 8
Engineered21 feature count: 21

Saved experimental stages:


,stage,stage_name,ecg_dimensions,auxiliary_dimensions,tabular_total_dimensions,auxiliary_features
0,1,ECG,255,0,255,None
1,2,ECG+6RR,255,6,261,6 immediate RR features
2,3,ECG+13Context,255,13,268,6 RR + 3 rhythm-history + 4 morphology-context
3,4,ECG+21,255,21,276,13 context + 8 explicit morphology


In [6]:
# ============================================================
# FINAL FEATURE-ENGINEERING VERIFICATION
# ============================================================

print(
    "=" * 70
)

print(
    "FINAL FEATURE ENGINEERING VERIFICATION"
)

print(
    "=" * 70
)


# ============================================================
# SECTION 1: VERIFY CANONICAL DS1 ROW ALIGNMENT
# ============================================================

assert (
    len(X_ds1_ecg)
    == len(X_ds1_rr6)
    == len(X_ds1_context13)
    == len(X_ds1_morphology8)
    == len(X_ds1_engineered21)
    == len(y_ds1)
    == len(ds1_metadata)
)


# ============================================================
# SECTION 2: VERIFY CANONICAL DS2 ROW ALIGNMENT
# ============================================================

assert (
    len(X_ds2_ecg)
    == len(X_ds2_rr6)
    == len(X_ds2_context13)
    == len(X_ds2_morphology8)
    == len(X_ds2_engineered21)
    == len(y_ds2)
    == len(ds2_metadata)
)


# ============================================================
# SECTION 3: VERIFY EXPECTED FEATURE DIMENSIONS
# ============================================================

assert (
    X_ds1_ecg.shape[1]
    == 255
)


assert (
    X_ds1_rr6.shape[1]
    == 6
)


assert (
    X_ds1_context13.shape[1]
    == 13
)


assert (
    X_ds1_morphology8.shape[1]
    == 8
)


assert (
    X_ds1_engineered21.shape[1]
    == 21
)


# ============================================================
# SECTION 4: VERIFY NO INFINITE VALUES
# ============================================================

assert not np.isinf(
    X_ds1_engineered21
).any()


assert not np.isinf(
    X_ds2_engineered21
).any()


# ============================================================
# SECTION 5: VERIFY LABELS STILL ALIGN WITH METADATA
# ============================================================

assert np.array_equal(
    y_ds1,

    ds1_metadata[
        "class"
    ]
    .to_numpy()
)


assert np.array_equal(
    y_ds2,

    ds2_metadata[
        "class"
    ]
    .to_numpy()
)


# ============================================================
# SECTION 6: VERIFY HEARTBEAT IDENTIFIERS REMAIN UNIQUE
# ============================================================

assert (
    ds1_metadata[
        "beat_uid"
    ]
    .is_unique
)


assert (
    ds2_metadata[
        "beat_uid"
    ]
    .is_unique
)


# ============================================================
# SECTION 7: VERIFY FIXED FOLDS HAVE NOT CHANGED
# ============================================================

fold_map = dict(
    zip(
        ds1_record_folds[
            "record_id"
        ],

        ds1_record_folds[
            "cv_fold"
        ]
    )
)


expected_cv_fold = (
    ds1_metadata[
        "record_id"
    ]
    .map(
        fold_map
    )
)


assert (
    expected_cv_fold
    .notna()
    .all()
)


assert np.array_equal(
    ds1_metadata[
        "cv_fold"
    ]
    .to_numpy(
        dtype=int
    ),

    expected_cv_fold
    .to_numpy(
        dtype=int
    )
)


# ============================================================
# SECTION 8: DISPLAY MISSING VALUES BY ENGINEERED FEATURE
# ============================================================

ds1_nan_count = np.isnan(
    X_ds1_engineered21
).sum(
    axis=0
)


ds2_nan_count = np.isnan(
    X_ds2_engineered21
).sum(
    axis=0
)


nan_summary = pd.DataFrame(
    {
        "feature":
            ENGINEERED21_FEATURE_NAMES,

        "DS1_nan_count":
            ds1_nan_count,

        "DS2_nan_count":
            ds2_nan_count,
    }
)


display(
    nan_summary
)


# ============================================================
# SECTION 9: DISPLAY FIRST FIVE ENGINEERED ROWS
# ============================================================

feature_preview = pd.DataFrame(
    X_ds1_engineered21[
        :5
    ],

    columns=
        ENGINEERED21_FEATURE_NAMES
)


feature_preview.insert(
    0,
    "class",

    y_ds1[
        :5
    ]
)


feature_preview.insert(
    0,
    "cv_fold",

    ds1_metadata[
        "cv_fold"
    ]
    .iloc[
        :5
    ]
    .to_numpy()
)


feature_preview.insert(
    0,
    "record_id",

    ds1_metadata[
        "record_id"
    ]
    .iloc[
        :5
    ]
    .to_numpy()
)


feature_preview.insert(
    0,
    "beat_uid",

    ds1_metadata[
        "beat_uid"
    ]
    .iloc[
        :5
    ]
    .to_numpy()
)


display(
    feature_preview
)


# ============================================================
# SECTION 10: DISPLAY FINAL EXPERIMENTAL DESIGN
# ============================================================

print(
    "\nFinal experimental stages:"
)


display(
    experiment_stage_manifest
)


print(
    "\nStage 1:"
)

print(
    "ECG = 255 dimensions"
)


print(
    "\nStage 2:"
)

print(
    "ECG + 6RR = 261 dimensions"
)


print(
    "\nStage 3:"
)

print(
    "ECG + 13Context = 268 dimensions"
)


print(
    "\nStage 4:"
)

print(
    "ECG + 21 engineered features = 276 dimensions"
)


# ============================================================
# SECTION 11: FINAL IMPUTATION REMINDER
# ============================================================

print(
    "\nIMPORTANT:"
)

print(
    "Missing context values are intentionally retained."
)

print(
    "Do not fit an imputer in Notebook 03."
)

print(
    "The imputer must be fitted only on the training "
    "portion of each fixed cross-validation fold."
)


print(
    "\nFixed DS1 folds were preserved successfully."
)


print(
    "\nNotebook 03 completed successfully."
)

FINAL FEATURE ENGINEERING VERIFICATION


,feature,DS1_nan_count,DS2_nan_count
0,rr_pre_sec,10,15
1,rr_post_sec,13,10
2,rr_local_median_sec,76,81
3,rr_pre_norm,76,81
4,rr_post_norm,89,91
5,rr_pre_post_ratio,23,25
6,rr_history_mean_sec,76,81
7,rr_history_std_sec,76,81
8,rr_history_cv,76,81
9,morph_corr_prev,22,22


,beat_uid,record_id,cv_fold,class,rr_pre_sec,rr_post_sec,rr_local_median_sec,rr_pre_norm,rr_post_norm,rr_pre_post_ratio,...,morph_corr_template,morph_dist_template,minimum_amplitude,maximum_amplitude,peak_to_peak,mean_absolute_amplitude,skewness,kurtosis,mean_absolute_derivative,maximum_absolute_derivative
0,101:2:396,101,2,N,0.869444,0.875000,NaN,NaN,NaN,0.993651,...,NaN,NaN,-0.795904,6.258281,7.054185,0.543148,4.142671,20.304508,0.078287,1.423668
1,101:3:711,101,2,N,0.875000,0.891667,NaN,NaN,NaN,0.981308,...,NaN,NaN,-0.758506,6.205700,6.964206,0.521132,4.211841,20.531260,0.074612,1.335435
2,101:4:1032,101,2,N,0.891667,0.933333,NaN,NaN,NaN,0.955357,...,NaN,NaN,-0.774007,6.134891,6.908897,0.546389,4.114536,19.583832,0.073442,1.298824
3,101:5:1368,101,2,N,0.933333,0.955556,0.875000,1.066667,1.092064,0.976744,...,0.992643,0.121124,-0.768894,6.163595,6.932489,0.545283,4.127740,19.845121,0.073490,1.298280
4,101:6:1712,101,2,N,0.955556,0.900000,0.883333,1.081761,1.018868,1.061728,...,0.970036,0.244349,-0.875602,6.011690,6.887292,0.533513,3.898877,18.528238,0.073506,1.248378



Final experimental stages:


,stage,stage_name,ecg_dimensions,auxiliary_dimensions,tabular_total_dimensions,auxiliary_features
0,1,ECG,255,0,255,None
1,2,ECG+6RR,255,6,261,6 immediate RR features
2,3,ECG+13Context,255,13,268,6 RR + 3 rhythm-history + 4 morphology-context
3,4,ECG+21,255,21,276,13 context + 8 explicit morphology



Stage 1:
ECG = 255 dimensions

Stage 2:
ECG + 6RR = 261 dimensions

Stage 3:
ECG + 13Context = 268 dimensions

Stage 4:
ECG + 21 engineered features = 276 dimensions

IMPORTANT:
Missing context values are intentionally retained.
Do not fit an imputer in Notebook 03.
The imputer must be fitted only on the training portion of each fixed cross-validation fold.

Fixed DS1 folds were preserved successfully.

Notebook 03 completed successfully.
